In [ ]:
import json #lee los archivos json

import pandas as pd #pandas

In [ ]:
VOTE_DECODE = {
    0: "SIN_DATO",
    1: "AFIRMATIVO",
    2: "NEGATIVO",
    3: "ABSTENCION",
    4: "AUSENTE",
    5: "PRESIDENTE",
} #diccionario

In [ ]:
with open("Datos/diputados.json", encoding="utf-8") as f:#abre el json y lo lee como diccionario
    data = json.load(f)

print(data.keys()) #sus claves
print(len(data["votaciones"])) #cuantas votaciones

dict_keys(['names', 'blocs', 'provinces', 'photo_ids', 'votaciones'])
2913


In [ ]:
filas = [] #lista vacia para juntar las filas

for votacion in data["votaciones"]: #recorre las 2293 votaciones 
    for voto in votacion["v"]:#adentro recorre cada voto
        id_diputado, id_bloque, id_provincia, codigo = voto #separa las cuatro vaariables

        filas.append({
            "votacion_id": votacion["id"],
            "titulo": votacion["t"],
            "fecha": votacion["d"],
            "resultado_general": votacion["r"],
            "diputado": data["names"][id_diputado] if id_diputado < len(data["names"]) else "SIN_DATO",
            "bloque": data["blocs"][id_bloque] if id_bloque < len(data["blocs"]) else "SIN_DATO",
            "provincia": data["provinces"][id_provincia] if id_provincia < len(data["provinces"]) else "SIN_DATO",
            "voto": VOTE_DECODE[codigo],
        }) #abre el diccionario de la fila,traduce las posiciones a nombres

print(len(filas))
print(filas[0])

748641
{'votacion_id': '1', 'titulo': 'Régimen previsional especial de carácter excepcional para los ex soldados combatientes de la guerra de Malvinas', 'fecha': '07/10/2015 - 14:26', 'resultado_general': 'AFIRMATIVO', 'diputado': 'ABDALA DE MATARAZZO, Norma Amanda', 'bloque': 'Frente Cívico por Santiago', 'provincia': 'Santiago del Estero', 'voto': 'AFIRMATIVO'}


In [ ]:
df = pd.DataFrame(filas) #lista a tabla cada clave del diccionario pasa a ser una columna

df.head()

,votacion_id,titulo,fecha,resultado_general,diputado,bloque,provincia,voto
0,1,Régimen previsional especial de carácter excep...,07/10/2015 - 14:26,AFIRMATIVO,"ABDALA DE MATARAZZO, Norma Amanda",Frente Cívico por Santiago,Santiago del Estero,AFIRMATIVO
1,1,Régimen previsional especial de carácter excep...,07/10/2015 - 14:26,AFIRMATIVO,"ABRAHAM, Alejandro",Frente para la Victoria - PJ,Mendoza,AFIRMATIVO
2,1,Régimen previsional especial de carácter excep...,07/10/2015 - 14:26,AFIRMATIVO,"AGUAD, Oscar Raúl",Unión Cívica Radical,Córdoba,AFIRMATIVO
3,1,Régimen previsional especial de carácter excep...,07/10/2015 - 14:26,AFIRMATIVO,"AGUILAR, Lino Walter",Compromiso Federal,San Luis,AUSENTE
4,1,Régimen previsional especial de carácter excep...,07/10/2015 - 14:26,AFIRMATIVO,"ALEGRE, Gilberto Oscar",Frente Renovador,Buenos Aires,AFIRMATIVO


In [ ]:
nuevos = pd.read_csv("Datos/votos_nuevos.csv", encoding="utf-8-sig") #lee las nuevas votaciones 

df["votacion_id"] = df["votacion_id"].astype(int) #pasa a texto a numeros
df = pd.concat([df, nuevos], ignore_index=True)# vuelve numerar las filas 

print("Nuevas:", len(nuevos), "| Total:", len(df)) #cuantas votaciones se sumaron

Nuevas: 20303 | Total: 768944


In [ ]:
df["voto"].value_counts() #cueenta cuantas veces aparece cada voto.

voto
AFIRMATIVO    433663
AUSENTE       177149
NEGATIVO      137720
ABSTENCION     16662
PRESIDENTE      2967
SIN_DATO         783
Name: count, dtype: int64

In [ ]:
df.to_csv("Datos/votos_expandidos.csv", index=False, encoding="utf-8-sig") #guarda el csv

print("Guardado:", len(df), "filas")

Guardado: 768944 filas


In [9]:
df = pd.read_csv("Datos/votos_expandidos.csv", encoding="utf-8-sig")

print("Filas:", len(df))

Filas: 768944


In [ ]:
df["fecha_dt"] = pd.to_datetime(df["fecha"], format="%d/%m/%Y - %H:%M", errors="coerce") #Texto → fecha. format describe el formato: día/mes/año - hora:minutos. errors="coerce": lo que no se pueda convertir queda vacío (NaT) en vez de dar error.

print("Fechas inválidas:", df["fecha_dt"].isna().sum()) #cuantos no se pudieron convertir 
print("Rango:", df["fecha_dt"].min(), "a", df["fecha_dt"].max()) #primera y ultima fecha 

Fechas inválidas: 0
Rango: 1993-06-23 18:26:00 a 2026-10-22 03:28:00


In [ ]:
print("Duplicados diputado-votación:", df.duplicated(subset=["votacion_id", "diputado"]).sum()) #marca las filas donde se duplica el par diputado-votacion

Duplicados diputado-votación: 155


In [12]:
print("Diputado SIN_DATO:", (df["diputado"] == "SIN_DATO").sum())
print("Provincia SIN_DATO:", (df["provincia"] == "SIN_DATO").sum())
print("Bloques vacíos:", df["bloque"].isna().sum())
print("Votos SIN_DATO o PRESIDENTE:", df["voto"].isin(["SIN_DATO", "PRESIDENTE"]).sum())

Diputado SIN_DATO: 0
Provincia SIN_DATO: 990
Bloques vacíos: 761
Votos SIN_DATO o PRESIDENTE: 3750


In [ ]:
df = df[~df["voto"].isin(["SIN_DATO", "PRESIDENTE"])] #no quedan las filas cuyo voto no esta en la lista
print(len(df))

765194


In [ ]:
df = df[df["diputado"] != "LEGISLADOR, A DESIGNAR"] #elimino las filas de la banca vacante
print(len(df))

764389


In [ ]:
provincia_habitual = (
    df[df["provincia"] != "SIN_DATO"] #toma solo las filas con provincia
    .groupby("diputado")["provincia"] #agrupa pordiputado y mira su columna de provincia 
    .agg(lambda s: s.mode()[0]) #toma la moda, su provincia mas frecuente
)

falta = df["provincia"] == "SIN_DATO" #marca las filas a recuperar
df.loc[falta, "provincia"] = df.loc[falta, "diputado"].map(provincia_habitual) #completa cada nombre segun la provincia

print("Sin recuperar:", df["provincia"].isna().sum()) #chequeo que no queda ninguna vacia

Sin recuperar: 0


In [ ]:
bloque_habitual = (
    df[df["bloque"].notna()] #es verdadero si el bloque no esta vacio
    .groupby("diputado")["bloque"] #agrupa por diputado
    .agg(lambda s: s.mode()[0]) #su bloque mas frecuente
)

falta = df["bloque"].isna() #filas sin bloque
df.loc[falta, "bloque"] = df.loc[falta, "diputado"].map(bloque_habitual) #las completa

print("Sin recuperar:", df["bloque"].isna().sum())

Sin recuperar: 0


In [ ]:
print("Filas:", len(df)) #filas
print("Duplicados:", df.duplicated(subset=["votacion_id", "diputado"]).sum()) #duplicados
print("Diputados:", df["diputado"].nunique()) #diputados distintos
print(df["voto"].value_counts()) #votos por tipo

df.to_csv("Datos/votos_limpios.csv", index=False, encoding="utf-8-sig") #guarda la tabla limpia

Filas: 764389
Duplicados: 0
Diputados: 2070
voto
AFIRMATIVO    433663
AUSENTE       176345
NEGATIVO      137719
ABSTENCION     16662
Name: count, dtype: int64


In [ ]:
df["anio"] = df["fecha_dt"].dt.year #saca el año de la fecha

bloques = (
    df.groupby("bloque") #agrupa por bloque
    .agg(
        votos=("voto", "size"),
        diputados=("diputado", "nunique"),
        desde=("anio", "min"),
        hasta=("anio", "max"),
    ) #clacula votos,diputados distintos, primer y ultimo año
    .sort_values("votos", ascending=False) #ordena de mas votos a menos
    .reset_index() #convierte nombre de bloque en columna
)

print("Bloques:", len(bloques))
bloques.head(20) #los 20 bloques mas grandes 

Bloques: 308


,bloque,votos,diputados,desde,hasta
0,Frente para la Victoria - PJ,173460,448,2005,2019
1,Unión Cívica Radical,105957,476,1993,2022
2,Justicialista,70262,517,1993,2021
3,Union Por La Patria,40420,137,2024,2026
4,Pro,39054,120,2017,2026
5,Frente De Todos,37392,181,2006,2023
6,La Libertad Avanza,23918,111,2021,2026
7,Unión PRO,15785,52,2014,2017
8,Coalicion Civica,10931,64,2010,2026
9,Partido Socialista,10818,40,2002,2018


In [ ]:
bloques["porcentaje_acumulado"] = (bloques["votos"].cumsum() / bloques["votos"].sum() * 100).round(1) #cuanto porcentaje acumulan los bloques

bloques.to_csv("Datos/bloques.csv", index=False, encoding="utf-8-sig") #guarda la lista

bloques[["bloque", "votos", "desde", "hasta", "porcentaje_acumulado"]].head(30) #mustra los primeros 30

,bloque,votos,desde,hasta,porcentaje_acumulado
0,Frente para la Victoria - PJ,173460,2005,2019,22.7
1,Unión Cívica Radical,105957,1993,2022,36.6
2,Justicialista,70262,1993,2021,45.7
3,Union Por La Patria,40420,2024,2026,51.0
4,Pro,39054,2017,2026,56.1
5,Frente De Todos,37392,2006,2023,61.0
6,La Libertad Avanza,23918,2021,2026,64.2
7,Unión PRO,15785,2014,2017,66.2
8,Coalicion Civica,10931,2010,2026,67.7
9,Partido Socialista,10818,2002,2018,69.1


In [ ]:
clasificacion = pd.read_csv("Datos/clasificacion_bloques.csv", encoding="utf-8-sig") #lee la tabl bloque

print(len(clasificacion)) #cuantos bloques calificados
clasificacion["macrocoalicion"].value_counts() #cuantos por macroalicion

307


macrocoalicion
PROVINCIALES    78
PERONISMO       65
IZQUIERDA       53
OTROS           40
PRO             32
RADICALISMO     31
LLA              8
Name: count, dtype: int64

In [ ]:
df["bloque"] = df["bloque"].str.replace(r"\s+", " ", regex=True).str.strip() #limpia los espacios
clasificacion["partido"] = clasificacion["partido"].str.replace(r"\s+", " ", regex=True).str.strip()

print(df["bloque"].nunique())

307


In [ ]:
df = df.merge(clasificacion, left_on="bloque", right_on="partido", how="left") #une las tablas busca el bloque en la columna partido y le pega su macrocoalicion
df = df.drop(columns="partido") #borra columnas repetidas

print("Filas:", len(df))
print("Sin macrocoalición:", df["macrocoalicion"].isna().sum())

Filas: 764389
Sin macrocoalición: 0


In [23]:
print(df["macrocoalicion"].value_counts())

df.to_csv("Datos/votos_macrocoaliciones.csv", index=False, encoding="utf-8-sig")

macrocoalicion
PERONISMO       391969
RADICALISMO     157953
PRO              73618
PROVINCIALES     71887
IZQUIERDA        36298
LLA              25124
OTROS             7540
Name: count, dtype: int64


In [ ]:
df = df.sort_values(["fecha_dt", "votacion_id"]).reset_index(drop=True) #ordena por fecha y numero de votacion

df["mes"] = df["fecha_dt"].dt.month

In [25]:
df["es_afirmativo"] = (df["voto"] == "AFIRMATIVO").astype(int)
df["es_negativo"] = (df["voto"] == "NEGATIVO").astype(int)
df["es_abstencion"] = (df["voto"] == "ABSTENCION").astype(int)
df["es_ausente"] = (df["voto"] == "AUSENTE").astype(int)

In [ ]:
grupo = df.groupby("diputado")

df["total_votos_previos"] = grupo.cumcount() #ordeno los votos previos de los diputados

In [ ]:
pares = {
    "es_afirmativo": "tasa_afirmativos_previos",
    "es_negativo": "tasa_negativos_previos",
    "es_abstencion": "tasa_abstenciones_previas",
    "es_ausente": "tasa_ausencias_previas",
} #diccionario

for origen, destino in pares.items():
    previos = grupo[origen].cumsum() - df[origen]
    df[destino] = (previos / df["total_votos_previos"] * 100).fillna(0) #divide los votos anteriores y  los multiplica por 100

In [ ]:
ejemplo = df[df["diputado"] == df["diputado"].iloc[0]].head(5) #muestro ejemplo con un diputado

ejemplo[["fecha", "voto", "total_votos_previos", "tasa_afirmativos_previos", "tasa_ausencias_previas"]]

,fecha,voto,total_votos_previos,tasa_afirmativos_previos,tasa_ausencias_previas
0,23/06/1993 - 18:26,AFIRMATIVO,0,0.000000,0.000000
276,23/06/1993 - 18:41,AUSENTE,1,100.000000,0.000000
674,30/06/1993 - 18:21,AUSENTE,2,50.000000,50.000000
896,30/06/1993 - 20:24,AFIRMATIVO,3,33.333333,66.666667
1076,30/06/1993 - 20:36,AFIRMATIVO,4,50.000000,50.000000


In [ ]:
votaciones = df[["votacion_id", "fecha_dt"]].drop_duplicates().sort_values("fecha_dt") #lista de votaciones unicas por fecha

n = len(votaciones)
corte_train = int(n * 0.70) #posicion del 70%
corte_val = int(n * 0.85) #posicion del 85%

ids_train = votaciones.iloc[:corte_train]["votacion_id"] #el 70% mas antiguo va para entrenamiento
ids_val = votaciones.iloc[corte_train:corte_val]["votacion_id"] #el 15% a validacion 

df["split"] = "test" #marca todo como set
df.loc[df["votacion_id"].isin(ids_train), "split"] = "train" #cambia a train las listas de entrenamiento
df.loc[df["votacion_id"].isin(ids_val), "split"] = "validation" #lo que queda para entrenamiento es el 15% mas recientes

print(df["split"].value_counts())
df.groupby("split")["fecha_dt"].agg(["min", "max"])

split
train         535319
test          114883
validation    114187
Name: count, dtype: int64


,min,max
split,,
test,2023-07-06 01:36:00,2026-10-22 03:28:00
train,1993-06-23 18:26:00,2018-03-21 18:44:00
validation,2018-03-21 19:17:00,2023-07-06 00:58:00


In [ ]:
print((df.groupby("split")["es_afirmativo"].mean() * 100).round(1)) #prporcion de afirmativos por conjunto

df.to_csv("Datos/dataset_features.csv", index=False, encoding="utf-8-sig") #guarda

split
test          57.3
train         55.5
validation    62.1
Name: es_afirmativo, dtype: float64


In [31]:
import pandas as pd

df = pd.read_csv("Datos/dataset_features.csv", encoding="utf-8-sig")

df["fecha_dt"] = pd.to_datetime(df["fecha_dt"])
df = df.sort_values(["fecha_dt", "votacion_id"]).reset_index(drop=True)

print(len(df))

764389


In [ ]:
grupo = df.groupby("diputado") #agrupor diputado

df["tasa_afirmativos_recientes"] = (
    grupo["es_afirmativo"]
    .transform(lambda s: s.shift(1).rolling(50, min_periods=1).mean() * 100)
    .fillna(0)
)

df["tasa_ausencias_recientes"] = (
    grupo["es_ausente"]
    .transform(lambda s: s.shift(1).rolling(50, min_periods=1).mean() * 100)
    .fillna(0)
)

In [33]:
con_mas_votos = df["diputado"].value_counts().index[0]
ejemplo = df[df["diputado"] == con_mas_votos]

print(con_mas_votos, len(ejemplo))
ejemplo[["fecha", "voto", "tasa_afirmativos_previos", "tasa_afirmativos_recientes"]].iloc[[0, 1, 2, 3, 4, 100, 300]]

CAMAÑO, GRACIELA 2314


,fecha,voto,tasa_afirmativos_previos,tasa_afirmativos_recientes
17,23/06/1993 - 18:26,AFIRMATIVO,0.000000,0.000000
268,23/06/1993 - 18:41,AFIRMATIVO,100.000000,100.000000
539,30/06/1993 - 18:21,AUSENTE,100.000000,100.000000
878,30/06/1993 - 20:24,AUSENTE,66.666667,66.666667
1172,30/06/1993 - 20:36,AUSENTE,50.000000,50.000000
67687,12/12/2000 - 14:53,NEGATIVO,37.000000,18.000000
137051,15/12/2005 - 18:31,AFIRMATIVO,61.666667,64.000000


In [34]:
df.to_csv("Datos/dataset_features.csv", index=False, encoding="utf-8-sig")

In [ ]:
df = pd.read_csv("Datos/dataset_features.csv", encoding="utf-8-sig", low_memory=False)
df["fecha_dt"] = pd.to_datetime(df["fecha_dt"]) #lee el csv

print(len(df))

764389


In [ ]:
df["origen"] = (
    df["titulo"] #crea columna a partir del titulo
    .str.extract(r"\b\d+\s*-\s*(PE|S|D|JGM)\s*-\s*\d{2,4}\b", expand=False) #busca expendiente
    .fillna("SIN_DATO")
)

df.drop_duplicates("votacion_id")["origen"].value_counts() #cuenta por votacion

origen
SIN_DATO    1517
D            592
S            424
PE           365
JGM           94
Name: count, dtype: int64

In [37]:
patron_mocion = "moci[oó]n|sobre tablas|apartamiento|vuelta a comisi|plan de labor"

df["es_mocion"] = df["titulo"].str.lower().str.contains(patron_mocion, regex=True).astype(int)

df.drop_duplicates("votacion_id")["es_mocion"].sum()

np.int64(433)

In [38]:
GOBIERNOS = [
    ("1989-07-08", "1999-12-10", ["PERONISMO"]),
    ("1999-12-10", "2001-12-21", ["RADICALISMO"]),
    ("2001-12-21", "2015-12-10", ["PERONISMO"]),
    ("2015-12-10", "2019-12-10", ["PRO", "RADICALISMO"]),
    ("2019-12-10", "2023-12-10", ["PERONISMO"]),
    ("2023-12-10", "2100-01-01", ["LLA"]),
]

df["es_oficialismo"] = 0

for desde, hasta, coaliciones in GOBIERNOS:
    en_periodo = (df["fecha_dt"] >= desde) & (df["fecha_dt"] < hasta)
    es_del_gobierno = df["macrocoalicion"].isin(coaliciones)
    df.loc[en_periodo & es_del_gobierno, "es_oficialismo"] = 1

print(df["es_oficialismo"].mean().round(3))

0.466


In [39]:
df.to_csv("Datos/dataset_features.csv", index=False, encoding="utf-8-sig")